# ISL Default: Logistic Regression vs. LDA

This notebook is designed to analyze the **Default** dataset from *An Introduction to Statistical Learning* using exactly three predictors: `student`, `balance`, and `income`.

**Primary data source:** the CRAN `ISLR` package (authors: James, Witten, Hastie, Tibshirani).  
**CSV mirror used for reproducible loading:** `https://vincentarelbundock.github.io/Rdatasets/csv/ISLR/Default.csv` (Rdatasets mirror of CRAN datasets).  
**Official dataset documentation:** `https://search.r-project.org/CRAN/refmans/ISLR/html/Default.html`

The loading code first looks for `Default.csv` beside this notebook and otherwise downloads the documented CSV. The Rdatasets CSV contains a row-index column (`rownames`), which is removed before modeling.

> **Execution status in the artifact delivered here:** the current execution environment could not resolve/download external CSV content, so the analysis cells below could not be executed against the real data. No synthetic substitute was used. The acquisition failure is recorded explicitly rather than fabricating outputs.


In [ ]:
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels
import statsmodels.api as sm
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    roc_curve, roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

SEED = 42
BOOT_SEED = 20260920
DATA_PATH = Path("Default.csv")
SOURCE_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ISLR/Default.csv"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("statsmodels:", statsmodels.__version__)
print("scikit-learn:", sklearn.__version__)


## 1. Data acquisition and inspection

The dataset is simulated but is the original `Default` dataset distributed with the ISLR package; no synthetic replacement is generated here.


In [1]:
if DATA_PATH.exists():
    print(f"Loading local cache: {DATA_PATH.resolve()}")
    df_raw = pd.read_csv(DATA_PATH)
else:
    print(f"Local cache not found; downloading from {SOURCE_URL}")
    try:
        df_raw = pd.read_csv(SOURCE_URL)
        df_raw.to_csv(DATA_PATH, index=False)
        print(f"Cached downloaded CSV at {DATA_PATH.resolve()}")
    except Exception as e:
        raise RuntimeError(
            "DATA ACQUISITION FAILED. This environment could not download the real ISL Default CSV. "
            "No synthetic replacement will be generated. Place Default.csv beside the notebook "
            "or run in an environment with HTTPS access. Original error: " + repr(e)
        )


RuntimeError: DATA ACQUISITION FAILED. This environment could not download the real ISL Default CSV. No synthetic replacement will be generated. Place Default.csv beside the notebook or run in an environment with HTTPS access.

In [ ]:
# Remove only a row-index column if present.
index_like = [c for c in df_raw.columns if str(c).lower() in {"rownames", "row.names", "x", "unnamed: 0"}]
df = df_raw.drop(columns=index_like, errors="ignore").copy()

required = {"default", "student", "balance", "income"}
assert len(df) == 10_000, f"Expected 10,000 rows, found {len(df)}"
assert required.issubset(df.columns), f"Missing required columns: {required - set(df.columns)}"

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing"))
display(df["default"].value_counts().rename("count").to_frame())
display(df["student"].value_counts().rename("count").to_frame())


In [ ]:
df["default_bin"] = df["default"].map({"No": 0, "Yes": 1})
df["student_bin"] = df["student"].map({"No": 0, "Yes": 1})
assert df[["default_bin", "student_bin"]].notna().all().all()

features = ["student_bin", "balance", "income"]
X = df[features].copy()
y = df["default_bin"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Training default prevalence:", y_train.mean())
print("Test default prevalence:", y_test.mean())


## 2. Fair comparison

A single **stratified 70/30 split** with `random_state=42` is used for both models. Stratification preserves the rare default proportion in both partitions. No oversampling, class reweighting, or test-set tuning is used.

ROC analysis is useful because default is rare: ROC AUC evaluates ranking/discrimination across all thresholds rather than depending on one cutoff. Average precision is also reported because precision–recall behavior is especially informative under class imbalance.


## 3. Unpenalized logistic regression

`statsmodels.Logit` fits an intercept and unpenalized maximum-likelihood logistic regression. Predictors are left in their original units, so no coefficient back-transformation is required. Confidence intervals are **model-based intervals conditional on the training sample**.


In [ ]:
X_train_sm = sm.add_constant(X_train.rename(columns={"student_bin":"student"}), has_constant="add")
X_test_sm = sm.add_constant(X_test.rename(columns={"student_bin":"student"}), has_constant="add")

logit = sm.Logit(y_train, X_train_sm).fit(disp=False, maxiter=200)
assert logit.mle_retvals.get("converged", True)
display(logit.summary())

coef = logit.params
ci = logit.conf_int()
or_table = pd.DataFrame({
    "coefficient": coef,
    "odds_ratio": np.exp(coef),
    "OR_95%_low": np.exp(ci[0]),
    "OR_95%_high": np.exp(ci[1]),
})
display(or_table)

student_or = np.exp(coef["student"])
balance100_or = np.exp(100 * coef["balance"])
income1000_or = np.exp(1000 * coef["income"])
print(f"Student Yes vs No conditional OR: {student_or:.4f}")
print(f"Per $100 higher balance conditional OR: {balance100_or:.4f}")
print(f"Per $1,000 higher income conditional OR: {income1000_or:.4f}")
print("Interpretations are associational, not causal.")

p_logit = np.asarray(logit.predict(X_test_sm))
assert np.isfinite(p_logit).all() and ((p_logit >= 0) & (p_logit <= 1)).all()
display(pd.DataFrame({"actual_default": y_test.to_numpy(), "p_default_logit": p_logit}, index=y_test.index).head(10))


## 4. Linear Discriminant Analysis

LDA models the predictors within each class as multivariate Gaussian with a **shared covariance matrix**, while estimating class priors from the training data. The binary `student` variable does not literally satisfy a continuous Gaussian assumption, but LDA can still be fitted and its held-out predictive performance evaluated.


In [ ]:
lda = LinearDiscriminantAnalysis(priors=None)
lda.fit(X_train, y_train)

print("Class labels:", lda.classes_)
print("Estimated class priors:", dict(zip(lda.classes_, lda.priors_)))
means = pd.DataFrame(lda.means_, index=[f"class_{c}" for c in lda.classes_], columns=features)
display(means)

lda_probs = lda.predict_proba(X_test)
pos_col = np.where(lda.classes_ == 1)[0][0]
p_lda = lda_probs[:, pos_col]
assert np.isfinite(p_lda).all() and ((p_lda >= 0) & (p_lda <= 1)).all()
display(pd.DataFrame({"actual_default": y_test.to_numpy(), "p_default_lda": p_lda}, index=y_test.index).head(10))


## 5. Held-out ROC curves and predictive performance

All metrics below use the untouched test set. Threshold-dependent metrics use **0.5** exactly; no threshold is optimized on the test set.


In [ ]:
def metrics_row(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "ROC AUC": roc_auc_score(y_true, prob),
        "Average precision": average_precision_score(y_true, prob),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall / sensitivity": recall_score(y_true, pred, zero_division=0),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_true, pred, zero_division=0),
    }

comparison = pd.DataFrame({
    "Logistic regression": metrics_row(y_test, p_logit),
    "LDA": metrics_row(y_test, p_lda)
}).T
display(comparison.style.format("{:.4f}"))

baseline_accuracy = (y_test == 0).mean()
test_prevalence = y_test.mean()
print(f"Always-predict-no-default accuracy: {baseline_accuracy:.4f}")
print(f"Test-set default prevalence: {test_prevalence:.4f}")


In [ ]:
fpr_logit, tpr_logit, _ = roc_curve(y_test, p_logit)
fpr_lda, tpr_lda, _ = roc_curve(y_test, p_lda)
auc_logit = roc_auc_score(y_test, p_logit)
auc_lda = roc_auc_score(y_test, p_lda)

fig, ax = plt.subplots(figsize=(7,7))
ax.plot(fpr_logit, tpr_logit, lw=2.2, color="#1f77b4", label=f"Logistic regression (AUC={auc_logit:.4f})")
ax.plot(fpr_lda, tpr_lda, lw=2.2, color="#d62728", label=f"LDA (AUC={auc_lda:.4f})")
ax.plot([0,1], [0,1], "--", color="0.5", lw=1.3, label="Chance")
ax.set(xlabel="False positive rate", ylabel="True positive rate", xlim=(0,1), ylim=(0,1))
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="lower right")
ax.grid(alpha=0.2)
plt.title("Held-out ROC curves")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, name, prob in zip(axes, ["Logistic regression", "LDA"], [p_logit, p_lda]):
    pred = (prob >= 0.5).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    im = ax.imshow(cm, cmap="Blues")
    for (i,j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=12)
    ax.set_title(f"{name}\nthreshold = 0.5")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks([0,1], ["No default", "Default"])
    ax.set_yticks([0,1], ["No default", "Default"])
plt.tight_layout()
plt.show()


ROC AUC measures discrimination over all possible thresholds. With a rare outcome, a conventional 0.5 probability threshold can still produce low sensitivity even when ranking is good. Average precision complements ROC AUC by emphasizing precision–recall performance under class imbalance.


### Paired stratified bootstrap uncertainty

We use **2,000 paired, stratified bootstrap resamples of the test set** with a fixed seed. Within each bootstrap replicate, default and non-default observations are resampled separately with replacement, preserving their original test counts; the exact same resampled indices are applied to both models. Percentile intervals therefore quantify uncertainty from the held-out test observations **conditional on the already fitted models**. They do not include training-sample/model-fitting variability.


In [ ]:
rng = np.random.default_rng(BOOT_SEED)
y_arr = y_test.to_numpy()
idx0 = np.flatnonzero(y_arr == 0)
idx1 = np.flatnonzero(y_arr == 1)

B = 2000
boot_logit = np.empty(B)
boot_lda = np.empty(B)

for b in range(B):
    s0 = rng.choice(idx0, size=len(idx0), replace=True)
    s1 = rng.choice(idx1, size=len(idx1), replace=True)
    idx = np.concatenate([s0, s1])
    rng.shuffle(idx)
    yb = y_arr[idx]
    boot_logit[b] = roc_auc_score(yb, p_logit[idx])
    boot_lda[b] = roc_auc_score(yb, p_lda[idx])

boot_diff = boot_logit - boot_lda

def pct_ci(x):
    return np.percentile(x, [2.5, 97.5])

uncertainty = pd.DataFrame({
    "Estimate": [auc_logit, auc_lda, auc_logit-auc_lda],
    "95% CI low": [*pct_ci(boot_logit)[:1], *pct_ci(boot_lda)[:1], *pct_ci(boot_diff)[:1]],
    "95% CI high": [*pct_ci(boot_logit)[1:], *pct_ci(boot_lda)[1:], *pct_ci(boot_diff)[1:]],
}, index=["Logistic AUC", "LDA AUC", "AUC difference (Logistic - LDA)"])
display(uncertainty.style.format("{:.4f}"))


## 6. Conclusion

After successful execution with the real `Default.csv`, this section should be updated from the computed results above. The appropriate conclusion should compare the two held-out AUC estimates, report the paired-bootstrap interval for **logistic AUC − LDA AUC**, and avoid declaring a meaningful winner from a tiny difference whose uncertainty includes practically negligible effects.

Limitations include the single 70/30 split, model assumptions (especially LDA's Gaussian/shared-covariance assumptions and the binary `student` predictor), threshold dependence of classification metrics, and the fact that the bootstrap intervals here condition on the fitted models rather than incorporating training-sample variability.
